# アンブレラサンプリングの初期構造の選択

これまでの工程でDeca-alanineを引き伸ばしたデータ（Steered MDの軌跡）が得られました。
続くアンブレラサンプリングでは特定の距離 (例えば d = 13.0Å, 13.5Å, ...)を中心とした独立したシミュレーションを多数実行します。

このノートブックではSteered MDのトラジェクトリーデータから、各アンブレラウインドウがターゲットとする距離に最も近い構造を初期構造として抜き出します。

## Step 1. ライブラリのimport

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

from ase.io import write, Trajectory

## Step2. 構造ファイルの読み込み

In [ ]:
# トラジェクトリーファイルのディレクトリ
traj_dir = "./output/03_steered_md"

# 出力ディレクトリ
out_dir  = "./input/04_umbrella_sampling"
os.makedirs(out_dir, exist_ok=True)

# 反応座標のトラジェクトリを読み込む
cv_traj = []
cv_range = np.arange(13.0, 33.1, 0.5) # CVの範囲と刻みを設定する

for cv in cv_range:
    cv_str = f'{cv:.2f}'
    cv_traj.append(np.loadtxt(f"{traj_dir}/cv_{cv_str}/COLVAR_{cv_str}")[1:, 1])

cv_traj = np.array(cv_traj)
cv_traj.shape

## Step 3. 各アンブレラウィンドウの初期構造を取り出す

設定したターゲット距離 (cv)について、対応するSteered MDのトラジェクトリーファイルを読み込み、ターゲット距離との差が最も小さいスナップショットを選びます。

In [21]:
# Prepare Initial Structures

print(f"  i,     cv,  cv of struct.")
for i, cv in enumerate(cv_range):
    diff = np.abs(cv - cv_traj[i])
    jmin = np.argmin(diff)
    print(f"{i:3d},  {cv:.2f},  {cv_traj[i][jmin]:.2f}")

    cv_str = f'{cv:.2f}'
    atoms = Trajectory(f'{traj_dir}/cv_{cv_str}/md-dyn.traj') 
    write(f'{out_dir}/initial_cv_{cv:.2f}.xyz', atoms[jmin])

  i,     cv,  cv of struct.
  0,  13.00,  13.44
  1,  13.50,  13.46
  2,  14.00,  14.00
  3,  14.50,  14.51
  4,  15.00,  14.96
  5,  15.50,  15.57
  6,  16.00,  16.04
  7,  16.50,  16.51
  8,  17.00,  16.97
  9,  17.50,  17.51
 10,  18.00,  18.02
 11,  18.50,  18.53
 12,  19.00,  18.70
 13,  19.50,  19.33
 14,  20.00,  19.67
 15,  20.50,  20.25
 16,  21.00,  20.84
 17,  21.50,  21.46
 18,  22.00,  22.01
 19,  22.50,  22.04
 20,  23.00,  22.64
 21,  23.50,  23.20
 22,  24.00,  23.21
 23,  24.50,  24.18
 24,  25.00,  24.71
 25,  25.50,  25.47
 26,  26.00,  25.98
 27,  26.50,  26.33
 28,  27.00,  27.02
 29,  27.50,  27.45
 30,  28.00,  28.01
 31,  28.50,  28.43
 32,  29.00,  28.92
 33,  29.50,  29.45
 34,  30.00,  30.10
 35,  30.50,  30.52
 36,  31.00,  31.00
 37,  31.50,  31.53
 38,  32.00,  31.98
 39,  32.50,  32.52
 40,  33.00,  32.40


## Next Steps

これで、アンブレラサンプリングに必要な一連の初期構造ファイル（`initial_cv_*.xyz`）が準備できました。
次の[ノートブック](./05_umbrella_sampling_ja.ipynb)では、実際にアンブレラサンプリングを実行します。アンブレラサンプリングが完了すると自由エネルギープロファイルを計算するフェーズへと進むことができます。